# Jinja

Jinja — текстовый шаблонизатор: есть некоторый текстовый шаблон (HTML, SQL, конфиг, письмо), содержащий плейсхолдеры и простую логику, а затем рендерите шаблон, передав данные (контекст) для заполнения плейсхолдеров

Главный use-case: параметризация конетнта. Мы выносим все изменяемое в отдельный конфиг, а сам контент делаем зависимым от этого конфига. Несколько сценариев использования:
- веб-приложения (Flask и др.)
- генерация HTML, писем, отчётов
- генерация SQL-кода (например, в dbt)
- генерация конфигурационных файлов (YAML, INI, JSON)

Как работает?<br>Текстовый шаблон - это аналог программы, которая должна быть выполнена. Запускаемый питоновский скрипт (в частности объект класса Environment) - это аналог компилятора. Парсинг программы в AST дерево происходит при запуске основной функции render(). Тогда же происходит проверка на доступность функций

Рендер шаблона из строки:

```python
tpl = Template("Hello, {{ user }}!")
print(tpl.render(user="Konstantin"))
```

Environment и загрузка шаблонов из файлов:
```python
env = Environment(
    loader=FileSystemLoader("templates"),
    autoescape=select_autoescape(["html", "xml"]),
)
tpl = env.get_template("index.html")
html = tpl.render(user={"name": "Ada"}, items=[1, 2, 3])
```

#### Базовый синтаксис

- {{ ... }} — вывести значение или выражение
- {% ... %} — управляющие конструкции
- {# ... #} — комментарии

```jinja
{# комментарий #}
<h1>Hello, {{ user.name }}!</h1>

{% if items %}
  <ul>
  {% for x in items %}
    <li>{{ x }}</li>
  {% endfor %}
  </ul>
{% else %}
  <p>Empty</p>
{% endif %}
```

#### Переменные

Какие данные можно использовать в {{ ... }}:
1. переданные аргументы в метод render()
2. глобальные переменные, зарегистрированные в `env.globals`
3. установленные в самом шаблоне
    - через set
    - через with 
4. фильтры `env.filters` / тесты `env.tests`
5. нативные конструкции, например, `loop.index`

Передавать можно что угодно, любой Python объект
```jinja
{{ user.name }}
{{ user["name"] }}
{{ items[0] }}
```

Комбинировать переменные в выражения:
```jinja
{{ price * qty }}
{{ "ok" if enabled else "no" }}
```

Если шаблон пытается использовать , не находит:
- при установленном редиме Undefined (стоит по умолчанию)
- при установленном режиме StrictUndefined

Присваивание:

```jinja
{% set title = "Report" %}
<h1>{{ title }}</h1>
```

#### Фильтры

В шаблоне можно размещать не только переменные и выражения над ними, но и применять Python функции для преобразования. Для этого они в шаблоне прицепляются через pipe "|"

С хера ли это называется "фильтры" не очень понял

Можно использовать стандартные Python-функции, и можно свои кастомные:<br>
```python
def slugify(s):
    return s.lower().replace(" ", "-")

env.filters["slugify"] = slugify
```

Пример использования фильтров в шаблоне<br>
```jinja
{{ name|lower }}
{{ items|join(", ") }}
{{ text|replace("a", "b") }}
{{ value|default("n/a") }}
```

#### Тесты: условное выполнение

По аналогии с трансформациями можно вставлять проверки - с их помощью можно делать условное выполнение: вывести содержимое, если проверка возвращает True

Синтаксис:<br>
{% if value is test_name %}  block contents {% endif %}<br>
{% if value is not test_name %}  block contents {% endif %}

test_name - имя булевой функции. В Jinga есть набор встроенных функций. Чтобы произвольная функция стала доступной для использования в шаблоне, её нужно разместить в поле tests объекта Environment

В примере ниже odd - питоновская булева функция<br>
```jinja
{% if user_id is odd %}
  odd
{% endif %}
```

Пример:

```python
def is_email(x):
    return isinstance(x, str) and "@" in x

env.tests["email"] = is_email
```


#### Ветвление (If-then-else)

if / elif / else:

```jinja
{% if score >= 90 %}A
{% elif score >= 80 %}B
{% else %}C
{% endif %}
```

for и объект loop:

```jinja
{% for item in items %}
  {{ loop.index }}: {{ item }}
{% endfor %}
```

Локальный контекст:

```jinja
{% with u=user %}
  {{ u.name }}
{% endwith %}
```


7. Макросы

```jinja
{% macro badge(text, kind="info") -%}
  <span class="badge badge-{{ kind }}">{{ text }}</span>
{%- endmacro %}

{{ badge("New", "success") }}
```

Импорт макросов:

```jinja
{% import "macros.html" as m %}
{{ m.badge("Hi") }}
```


8. Наследование и include

base.html:
```jinja
<!doctype html>
<html>
  <body>
    {% block content %}{% endblock %}
  </body>
</html>
```

page.html:
```jinja
{% extends "base.html" %}
{% block content %}
  <h1>{{ title }}</h1>
{% endblock %}
```

Include:
```jinja
{% include "partials/header.html" %}
```


9. Экранирование и безопасность

```jinja
{{ user_input|e }}
{{ trusted_html|safe }}
```


10. Управление пробелами

```jinja
{%- for x in items -%}
{{ x }}
{%- endfor -%}
```


11. Environment и настройки

```python
env.globals["app_name"] = "MyApp"
```


### Практические задачи и типовые сценарии

Сценарий 1. HTML-страница со списком объектов

```jinja
<ul>
{% for user in users %}
  <li>{{ user.name }} ({{ user.email }})</li>
{% endfor %}
</ul>
```

Сценарий 2. Генерация SQL-запроса

```jinja
SELECT *
FROM {{ table }}
WHERE date >= '{{ start_date }}'
{% if country %}
  AND country = '{{ country }}'
{% endif %}
```

Сценарий 3. Генерация YAML-конфига

```jinja
services:
{% for s in services %}
  - name: {{ s.name }}
    port: {{ s.port }}
{% endfor %}
```

Сценарий 4. Email-шаблон

```jinja
Hello {{ user.name }},

{% if overdue %}
Your payment is overdue by {{ days }} days.
{% else %}
Thank you for your payment!
{% endif %}
```

Сценарий 5. Переиспользование через макросы

```jinja
{% macro field(label, value) %}
{{ label }}: {{ value }}
{% endmacro %}

{{ field("Name", user.name) }}
{{ field("Email", user.email) }}
```


13. Sandbox и безопасность

Для рендера недоверенных шаблонов используйте sandboxed environment,
который ограничивает доступ к методам и атрибутам объектов.


14. Практические рекомендации

- Сложную логику держите в Python
- Используйте autoescape для HTML
- Для надёжности — строгий undefined
- Переиспользование: наследование + include + макросы
